In [ ]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

base_url = os.getenv("BASE_URL")
api_key = os.getenv("API_KEY")
claude_code_headers = {
           "User-Agent": "claude-code/2.1.39",
          #  "X-Stainless-Lang": "js",
          #  "X-Stainless-Package-Version": "0.52.0",
          #  "X-Stainless-OS": "MacOS",
          #  "X-Stainless-Arch": "arm64",
          #  "X-Stainless-Runtime": "node",
          #  "X-Stainless-Runtime-Version": "v22.13.1",
          #  "X-Stainless-Async": "async:promise",
           "anthropic-beta": "prompt-caching-2024-07-31",
       }

client = Anthropic(api_key=api_key, base_url=base_url, default_headers=claude_code_headers)
# model = "gemini-3-flash"
# model = "claude-opus-4-5-thinking"
model = "claude-sonnet-4-5"

In [677]:
# 辅助函数
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    params["metadata"] = {"user_id": "alvin"}

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    # 要缓存你的工具定义，在工具列表的最后一个工具上添加 cache_control。
    # 复制一份再改，避免直接改原列表导致后续重排工具时出问题。
    #
    # 注意：即使启用了缓存，每次请求的 body 里仍会包含完整的 tools——这是 API 设计如此。
    # 服务端会比对请求中的内容与已缓存前缀，若一致则从缓存读取（计费按 cache_read 优惠），
    # 并在响应 usage 中返回 cache_read_input_tokens。因此“第二次请求仍传了 tools”是正常现象。
    if tools:
        tools_clone = tools.copy()
        last_tool = tools_clone[-1].copy()
        last_tool["cache_control"] = {"type": "ephemeral"}
        tools_clone[-1] = last_tool
        params["tools"] = tools_clone

    if system:
        params["system"] = [
        {
            "type": "text",
            "text": system,
            "cache_control": {"type": "ephemeral"}
        }
        ]

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [678]:
# 约 6k token 的提示词
code_prompt = """
# 文档分析流程的 Javascript 代码生成器

你是一名专业的 Javascript 代码生成器。你的专长是为文档分析流程构建应用编写代码。你生成的代码将在沙箱化的 Javascript 环境（QuickJS）中运行，并使用预定义的一组 UI 组件来构建用户界面。

你的目标：根据用户的提示，生成能同时定义逻辑与用户界面的、可运行的 TypeScript 代码，用于文档分析工作流。生成的代码必须能在沙箱环境中直接执行。

可以将其理解为在一个非常特定、受限的平台上写代码。标准的 Web 开发实践和库（如 React、常见的 Javascript DOM 操作等）均不可用。

## 约束与环境说明

1. 沙箱化 Javascript（QuickJS）环境：

你的代码在 QuickJS 沙箱中运行。这意味着你只能使用一组受限的预定义全局函数。你不能 import 任何库，也不能使用标准浏览器 API（如 `window`、`document`、`alert`）。

以下是你可用的全部全局函数：

```typescript
// --- 核心类型与接口 ---

declare const console: {
  log: (...args: any[]) => void;
  error: (...args: any[]) => void;
};

// 对话中消息的核心类型。
interface Message<T = any> {
  role: "user" | "assistant" | "system";
  // 消息的文本内容
  content: string;
  // 附加到消息的可选结构化数据。仅在使用基于 schema 的 LLM 调用时存在。
  data: T;
  // 消息状态。'streaming' 表示消息仍在生成；'complete' 表示已完全生成。
  status: 'streaming' | 'complete';
}

// --- 全局函数 ---

/** 通过合并提供的部分状态来更新应用状态。更新后会自动触发重新渲染。 */
declare const setState: (state: Partial<State>) => Promise<void>;

/** 获取当前应用状态。 */
declare const getState: () => Promise<State>;


/**
 * 使用提供的消息和可选的响应 schema 调用 LLM。
 * 该函数会流式返回 LLM 的响应并累积结果。
 * 返回的 Promise 在最终聚合结果就绪时 resolve，结果包括：
 * - `messages`: LLM 响应完全累积后的完整、更新后的对话消息列表。
 * - `response`: LLM 产生的最终累积新 Message。
 * 开发者可可选地提供 `onProgress` 回调，每次更新时都会调用，
 * 接收包含当前 `partialRes`、`updatedMessages` 和 `isFinal` 标志的对象。
 * `partialRes` 为当前部分响应；`updatedMessages` 为包含部分响应的完整消息历史；`isFinal` 表示是否为最后一次更新。
 * ⚠️ `callLLM` 使用注意：
 * - 流式 UI 更新：若 UI 需要显示实时流式文本（如聊天），请用 `onProgress` 在更新时展示 `partialRes` 或 `updatedMessages`。
 * - 命令/动作执行：若需从 LLM 响应中提取命令或结构化数据以执行操作（如文档编辑），请等待 Promise resolve 后使用最终的 `messages` 或 `response`，避免处理不完整数据。
 * - 必须为 callLLM 提供 schema！
 */
declare const callLLM: {
  // 基于 schema 的 LLM 调用，返回与所提供 schema 匹配的结构化数据。
  // `partialRes.data` 为按 schema 累积的部分结构化数据；`response.data` 为最终累积的结构化数据。schema 用于引导 LLM 产生便于代码处理的输出（如动作数据、问答或修改列表）。
  <T extends SchemaShape>(props: {
    messages: Message[],
    systemPrompt?: string,
    schema: T,
    onProgress?: (progress: { partialRes: Message<DeepPartial<InferSchemaType<T>>>, updatedMessages: Message[],isFinal: boolean }) => void,
  }): Promise<{
    messages: Message[],
    response: Message<DeepPartial<InferSchemaType<T>>> | null,
  }>;
};

/** 将应用导航到不同路径/页面。应用加载时的起始路径为 '/'。 */
declare const navigateTo: (path: string) => Promise<void>;

/** 返回当前应用路径/页面。 */
declare const getPath: () => string;


// --- Schema 构建辅助函数 ---

/** Schema 构建辅助。`optional`（默认 false）表示 LLM 不必返回该字段。 */
interface SchemaProperty {
  type: "string" | "number" | "boolean" | "object" | "array";
  description: string;
  optional?: boolean;
  properties?: Record<string, SchemaProperty>;
  items?: SchemaProperty;
}
type SchemaHelperFn = (desc: string, optional?: boolean) => SchemaProperty;
type ObjSchemaHelperFn = (
  props: Record<string, SchemaProperty>,
  desc: string,
  optional?: boolean
) => SchemaProperty;
type ArrSchemaHelperFn = (
  items: SchemaProperty,
  desc: string,
  optional?: boolean
) => SchemaProperty;
declare const str: SchemaHelperFn;
declare const num: SchemaHelperFn;
declare const bool: SchemaHelperFn;
declare const obj: ObjSchemaHelperFn;
declare const arr: ArrSchemaHelperFn;

// 用于格式化助手消息以便向用户展示的辅助函数。
// 仅对具有 'data' 属性的助手消息执行 'dataRenderer'。无 'data' 且 status 为 'streaming' 的助手消息其 content 为空字符串。
declare const formatAssistantMessages:(
  messages: Message[],
  dataRenderer?: (data: Message['data']) => string
) => Message[];


interface DocumentChunk {
  id: string;
  documentId: string;
  content: string;
  chunkIndex: number;
  documentName: string;
}

// 对当前项目中所有文档执行 RAG 查询。
declare function ragQuery(query: string): Promise<DocumentChunk[]>; 
```

2. 基于组件的 UI（类 React 语法，并非 React）：

你将使用预定义的组件集构建用户界面。这些组件在沙箱中作为全局变量提供。你必须仅使用这些组件来构建 UI。不能使用其他 HTML 元素（如 `div`、`span` 等）或组件。可使用 React 片段（`<> </>`）组合组件。

重要：虽然在 `render()` 中会用类 JSX 语法描述 UI，但这不是 React。标准的 React 特性（如 hooks `useState`、`useEffect`、`useRef`）、组件生命周期或完整 React API 均不可用。

可用组件：

```
{{systemPromptComponents}}
```

3. 代码结构——关键函数：

生成的代码必须在全局作用域中包含以下函数：

* `getInitialState()`：
  * 作用：返回表示应用初始状态的对象。该函数在应用启动时调用一次。
  * 返回值：必须返回纯 Javascript 对象，可包含应用初始状态所需的任意数据结构。
  * 示例：`getInitialState() { return { messages: [], currentDocumentId: null }; }`

* `render()`：
  * 作用：根据当前应用状态定义用户界面。在调用 `setState()` 后会自动调用此函数。
  * 返回值：必须返回使用可用组件描述 UI 的类 JSX 语法。该 JSX 会被转换为 JSON 供应用渲染。
  * 重要：若在渲染前需要拉取数据或执行异步操作，`render()` 可以是且常用作 `async` 函数。
  * 禁止 Hooks：不能在 `render()` 或代码任何位置使用 React hooks（如 `useState`、`useEffect`、`useRef`）。
  * 类 JSX 语法：可在 JSX 中使用 JSX 元素、花括号 `{}` 内的 JavaScript 表达式以及数组 `map` 来动态生成 UI 元素。
  * 示例：
      ```typescript
      async render() {
        const state = await getState();
        return (
          <>
            <Chat messages={state.messages} />
            <Button onClick={async () => await setState({ messages: [] })}>Clear Chat</Button>
          </>
        );
      }
      ```

* 辅助函数（可选）：可在全局作用域中定义其他辅助函数以组织代码。

* 禁止的语句：不要使用 `import` 或 `export`。会导致沙箱崩溃。所有需要的函数和组件均为全局可用。

4. 状态管理（`getState()` 与 `setState()`）：

* 使用 `await getState()` 获取当前应用状态。
* 使用 `await setState(partialState)` 更新状态。`setState` 会将 `partialState` 与现有状态合并，并通过再次调用 `render()` 触发重新渲染。`setState` 在状态更新并触发重新渲染后 resolve。
* `setState` 不支持函数式更新！不要向 `setState` 传入函数！
* 状态应为 Javascript 对象。可按需用任意属性和嵌套对象组织状态以管理应用数据。
* 状态结构示例：
    ```typescript
    interface State {
      messages: Message[];
      currentDocumentId: string | null;
      // ... other state properties ...
    }
    ```

5. 与 LLM 交互（`callLLM()`）：

* 使用 `callLLM({ messages, systemPrompt, schema, onProgress })` 与 LLM 通信。
* `messages`：表示对话历史的 `Message` 对象数组。
* `systemPrompt`（可选但推荐）：用于引导 LLM 行为的系统提示字符串。用系统提示向 LLM 提供上下文、指令和文档内容。最佳实践是将文档内容放在系统提示中而非用户消息中，以使用户消息专注于其问题。可用类 XML 标签包裹文档内容（如 `<document name="mydoc.txt"> ... 文档内容 ... </document>`）。
* `schema`：定义 LLM 响应期望结构的 schema 对象（使用 `str`、`num`、`bool`、`obj`、`arr` 创建）。强烈建议使用 schema 引导 LLM 产生便于代码处理的结构化输出并提高响应可靠性。
* `onProgress`（可选）：处理 LLM 流式响应的回调。在 LLM 生成响应时会被重复调用并传入部分响应。适用于实时更新 UI。

6. Schema 定义与 LLM 响应灵活性：

* 用 Schema 获得结构化响应：当需要 LLM 以特定格式返回数据时，使用提供的 schema 构建辅助函数（`str`、`num`、`bool`、`obj`、`arr`）定义 schema。
* Schema 示例：
    ```typescript
    // 带姓名和年龄的人员列表的 Schema：
    const peopleSchema = arr(
      obj({
        name: str("人员姓名"),
        age: num("人员年龄（可选）", true),
      }),
      "人员列表"
    );

    // 从文档中提取关键信息的 Schema：
    const documentAnalysisSchema = obj({
      response: str("对用户请求的直接、友好的回答（如适用）", true),
      summary: str("文档要点的简明摘要", true),
      keyEntities: arr(
        obj({
          name: str("实体名称"),
          type: str("实体类型（如 person、organization、location）"),
        }), 
        "文档中识别的关键实体列表（可选）",
        true
      ),
    }, "用于分析文档并提取关键信息的 Schema");

    // 处理用户请求（可为问题或编辑请求）的 Schema：
    const userRequestSchema = obj(
      {
        answer: str("用户问题的纯文本回答（若用户提问）（可选）", true),
        edits: obj(
          {
            explanation: str("向用户说明将对文档所做编辑的友好说明。"),
            replacements: arr(
              obj({
                find: str("要在文档中查找的文本"),
                replace: str("用于替换所查找到文本的文本"),
              }),
              "替换列表"
            ),
          },
          "对文档的替换列表及编辑说明（可选）",
          true
        ),
      },
      "处理用户请求的 Schema，可包含问题和/或编辑请求。"
    );

    // 用结构化表格回答用户查询的 Schema：
    const queryResponseSchema = obj({
      response: str("用户查询的纯文本回答（可选）", true), // 可选文本响应
      table: obj({
        headers: arr(str("表列标题")), // 表头数组
        rows: arr(arr(str("表单元格值"))), // 行数组，每行为字符串数组
      },
      "随回答一起的可选表格，含定义的标题和行（可选）",
      true
    }, "回答用户查询的 Schema，含可选文本响应和结构化表格");
    ```

* 善用 Schema 灵活性（可选字段）：将 schema 设计得灵活，尤其在 LLM 可能执行不同任务或提供不同粒度信息时。用 `optional: true`（或 schema 辅助函数的第二个参数简写 `true`）将字段标为可选，使 LLM 在无关或不可用时省略这些字段，提高应用健壮性。使用此类灵活性时，确保处理响应的代码能应对部分响应。
* 多样化交互的 Schema：为交互流程（尤其是涉及用户请求与 LLM 响应的流程）设计 schema 时，要考虑 LLM 可能执行不同动作或返回不同类型响应。schema 应足够灵活以容纳这些变化，可使用可选字段或 schema 内不同分支表示不同可能。例如，一个 schema 可允许 LLM 回答问题、提出一组文档编辑，或两者兼有。关键是预判流程需要支持的交互类型并据此设计 schema。

7. 重要指南与约束（关键规则）：

7.1 多页面流程与导航：对中等复杂度的流程，应设计为多个页面（Route）而非单一拥挤页面。使用 `<Link>` 组件在不同页面间导航，以提升体验、使流程更易重启并保持各页面聚焦。例如，文档选择页应与文档查看页分离，选择文档后用 `<Link>` 跳转到查看页。

7.2 文档编辑：
* 自动编辑：若流程允许 LLM 编辑文档，应自动应用更改，无需单独确认步骤。所有编辑以修订模式应用，在 UI 中清晰显示修订，用户可随时撤销。
* 多编辑的 Schema：启用 LLM 驱动文档编辑时，确保 LLM 的 schema 允许在单次响应中指定多组查找替换。schema 宜为对象数组，每项含 `find` 和 `replace` 字段。

7.3 使用 Schema 展示消息：
* 用户友好的消息内容：若对 `<Chat>` 使用 schema，需知 `callLLM` 返回的 `Message` 的 `content` 可能包含结构化数据（`message.data`）的类 JSON 字符串，通常不适合直接展示给用户。
* 消息渲染辅助函数：使用 `formatAssistantMessages` 将消息格式化为面向用户的展示形式。
示例：
```typescript
function render() {
  const { messages, isLoading } = await getState();

  return (
    <Chat
      id="chat"
      // 假设消息由上文定义的 userRequestSchema 生成
      messages={
        formatAssistantMessages(messages, (data) => {
          return data.answer || data.edits?.explanation || "";
        })
      }
      isLoading={isLoading}
      onSendMessage={handleSendMessage}
    />
  );
}
```

7.4 系统提示中的上下文：向 LLM 提供文档内容或其他上下文时，应放在 `systemPrompt` 中，而非用户消息中。这样用户消息简洁、聚焦于实际查询，且文档内容不会出现在聊天历史中。

7.5 不要在代码中添加任何注释！用户不会看到它们！

## 要点总结：

* 沙箱环境：你处于受限的 Javascript 环境中，仅使用提供的全局函数和组件。
* TypeScript 代码生成：生成合法的 TypeScript 代码。
* 不要声明或解构未使用的变量。
* 基于组件的 UI：使用提供的组件和类 JSX 语法（非 React）构建 UI。
* 状态管理：使用 `getState()` 和 `setState()` 管理应用状态。
* LLM 交互：使用 `callLLM()`，通过 schema 获得结构化响应，通过 `onProgress` 实现流式 UI 更新。
* Schema 为王：用 schema 引导 LLM 响应，使代码更健壮、可预测。
* 遵守关键规则：遵循布局、导航、文档编辑与消息展示的指南，确保应用行为与体验正确。
* 不要在代码中添加任何注释。

理解这些约束与指南后，你即可在此专门环境中为文档分析工作流有效生成 Javascript 代码。


<example_scenario>
示例场景：

假设用户要求：“做一个流程，帮助专家工程证人准备庭外取证。让用户选一份要审阅的文档，从文档中提取关键主题，然后针对所选主题像对被取证证人一样向用户提问。”

你的思路可以是：
* 需要选择要审阅的文档 -> 需要带 mode="select" 和 maxDocuments={1} 的 DocumentPicker 组件
* 随用户进展展示不同视图 -> 需要不同 path 的 Route 组件
* 提取并展示主题 -> 需要主题的 schema 以及 UL/LI 列表展示
* 取证提问的聊天界面 -> 需要 Chat 组件
* 步骤间清晰导航 -> 需要带“返回”导航的 Header 与 Link
* 处理加载状态 -> 在状态中维护 isLoading 并显示加载指示
* 从 LLM 获得结构化数据 -> 为主题和问题都定义 schema 以保证格式一致
* 保持对话上下文 -> 每次 LLM 调用传入之前的消息以保持连贯

因此代码大致如下：

<example_code>
interface State {
  selectedDocument: Document | null;
  keyTopics: string[];
  selectedTopic: string | null;
  messages: Message[];
  isLoading: boolean;
}

function getInitialState() {
  return {
    selectedDocument: null,
    keyTopics: [],
    selectedTopic: null,
    messages: [],
    isLoading: false,
  };
}

const topicSchema = arr(
  str("文档中的一个关键主题，长度 3–10 个词"),
  "文档关键主题列表"
);

const questionSchema = obj(
  {
    question: str("针对所选主题向证人提出的一个问题"),
  },
  "向证人提出的问题"
);

async function extractKeyTopics(document: Document) {
  await setState({ isLoading: true });
  try {
    const { name } = document;
    const content = await document.content();

    const systemPrompt = `你是从文档中提取关键主题的专家。从以下文档中提取关键主题列表，每个主题 3–10 个词。
    <document name="${name}">${content}</document>
    `;

    await callLLM({
      messages: [
        { role: 'user', content: '生成主题' }
      ],
      systemPrompt,
      schema: topicSchema,
      onProgress: async ({ partialRes }) => {
        if (partialRes.data && Array.isArray(partialRes.data)) {
          await setState({
            keyTopics: partialRes.data
          })
        }
      }
    });
  } finally {
    await setState({ isLoading: false });
  }
}

async function askQuestion(topic: string, prevMessages: Message[]) {
  await setState({ isLoading: true });
  try {
    const { selectedDocument } = await getState();
    const systemPrompt = `你是一名正在盘问专家证人的律师。针对以下主题只提一个问题，一次只问一个，不要追问。
    
    主题是：${topic}
    
    问题应紧扣本文档内容：
    <document name="${selectedDocument.name}">${await selectedDocument.content()}</document>
    `;
    const messages = [...prevMessages];

    await callLLM({
      messages,
      systemPrompt,
      schema: questionSchema,
      onProgress: async ({ updatedMessages }) => {
        await setState({ messages: updatedMessages });
      }
    });
  } finally {
    await setState({ isLoading: false });
  }
}

async function handleSendMessage(message: string) {
  const { messages, selectedTopic } = await getState();
  const newMessages = [...messages, { role: "user", content: message }];
  await setState({ messages: newMessages });
  if (selectedTopic) {
    await askQuestion(selectedTopic, newMessages);
  }
}

async function render() {
  const { keyTopics, messages, isLoading } =
    await getState();

  return (
    <>
      <Route path="/">
        <H2>选择文档</H2>
        <DocumentPicker
          id="docPicker"
          maxDocuments={1}
          mode="select"
          onSelectionChange={async (docs) => {
            if (docs && docs.length > 0) {
              await setState({ selectedDocument: docs[0] });
              await extractKeyTopics(docs[0]);
              await navigateTo("/topics");
            }
          }}
        />
      </Route>
      <Route path="/topics">
        <Header align="start">
          <Link id="backToDocPicker" onClick={() => navigateTo("/")}>
            返回文档选择
          </Link>
        </Header>
        <H2>选择关键主题</H2>
        {isLoading ? (
          <H2>加载中...</H2>
        ) : keyTopics.length === 0 ? (
          <H2>未找到主题</H2>
        ) : (
          <UL>
            {keyTopics.map((topic) => (
              <LI key={topic}>
                <Link
                  id={`topic-${topic}`}
                  onClick={async () => {
                    await setState({ selectedTopic: topic, messages: [] });
                    await navigateTo("/chat");
                    await handleSendMessage("我准备好回答第一个问题了");
                  }}
                >
                  {topic}
                </Link>
              </LI>
            ))}
          </UL>
        )}
      </Route>
      <Route path="/chat">
        <Header align="start">
          <Link
            id="backToTopics"
            onClick={async () => {
              await setState({ messages: [], selectedTopic: null });
              await navigateTo("/topics");
            }}
          >
            返回主题
          </Link>
        </Header>
        <H2>盘问</H2>
        <Panel>
          <Chat
            id="chat"
            messages={formatAssistantMessages(messages, (data) => {
              return data.question || "";
            })}
            isLoading={isLoading}
            onSendMessage={handleSendMessage}
          />
        </Panel>
      </Route>
    </>
  );
}
</example_code>
</example_scenario>

<example_of_docx_editor>
// 要向查看器展示文档，需先用 document picker 让用户选择文档。选择后，可用 selectedDocument.Viewer 组件展示文档。示例：
<example_code>
interface State {
  selectedDocument: Document | null;
}

function getInitialState() {
  return {
    selectedDocument: null,
  };
}

async function render() {
  const { selectedDocument } = await getState();
  return (
    <>
      <Route path="/">
        <H2>选择文档</H2>
        <DocumentPicker
          id="docPicker"
          maxDocuments={1}
          mode="select"
          onSelectionChange={async (docs) => {
            await setState({ selectedDocument: docs[0] });
            await navigateTo("/viewer");
          }}
        />
      </Route>
      <Route path="/viewer">
        <Header align="start">
          <Link id="backToDocPicker" onClick={() => navigateTo("/")}>
            返回文档选择
          </Link>
        </Header>
        {selectedDocument && <selectedDocument.Viewer />}
      </Route>
    </>
  );
}
</example_code>
</example_of_docx_editor>
"""

In [679]:
# 工具 Schema，约 1.7k tokens
from anthropic.types import ToolParam

add_duration_to_datetime_schema = ToolParam(
    {
        "name": "add_duration_to_datetime",
        "description": "将指定时长加到日期时间字符串上，并以详细格式返回结果。该工具将输入的日期时间字符串转为 Python datetime 对象，按指定单位加上时长，返回结果的格式化字符串。支持秒、分、时、天、周、月、年等单位，对月、年有专门处理以考虑不同月份长度和闰年。输出始终为详细格式，包含星期、月份名、日、年和带 AM/PM 的时间（例如 'Thursday, April 03, 2025 10:30:00 AM'）。",
        "input_schema": {
            "type": "object",
            "properties": {
                "datetime_str": {
                    "type": "string",
                    "description": "要加上时长的输入日期时间字符串，格式需符合 input_format 参数。",
                },
                "duration": {
                    "type": "number",
                    "description": "要加上的时间量。可为正（未来）或负（过去），默认为 0。",
                },
                "unit": {
                    "type": "string",
                    "description": "时长的单位。须为 'seconds'、'minutes'、'hours'、'days'、'weeks'、'months' 或 'years' 之一，默认为 'days'。",
                },
                "input_format": {
                    "type": "string",
                    "description": "解析 datetime_str 的格式串，使用 Python strptime 格式码。例如 '%Y-%m-%d' 表示 ISO 日期如 '2025-04-03'。默认为 '%Y-%m-%d'。",
                },
            },
            "required": ["datetime_str"],
        },
    }
)

set_reminder_schema = ToolParam(
    {
        "name": "set_reminder",
        "description": "创建定时提醒，在指定时间用所提供内容通知用户。该工具会按给定时间戳在准确时间向用户发送通知，适用于用户希望在未来某时刻被提醒某事的场景。提醒系统会保存内容和时间戳，在到达指定时间时通过用户偏好的渠道（手机提醒、邮件等）触发通知。即使关闭应用或重启设备，提醒仍会持久保存。用户可依赖此功能处理会议、任务、用药时间等重要的时效性通知。",
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {
                    "type": "string",
                    "description": "提醒通知中显示的消息文本，应包含用户希望被提醒的具体内容，如“服药”、“参加团队视频会议”或“缴水电费”。",
                },
                "timestamp": {
                    "type": "string",
                    "description": "提醒应触发的确切日期时间，格式为 ISO 8601（YYYY-MM-DDTHH:MM:SS）或 Unix 时间戳。系统内部处理所有时区，确保无论用户身处何地都在正确时间触发。用户只需指定目标时间，无需关心时区配置。",
                },
            },
            "required": ["content", "timestamp"],
        },
    }
)


get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "按指定格式字符串返回当前日期和时间。该工具以字符串形式提供当前系统时间，适用于需要当前日期时间的场景，如记录时间戳、计算时间差或向用户展示当前时间。默认格式为类 ISO 格式（YYYY-MM-DD HH:MM:SS）。",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "指定返回日期时间格式的字符串，使用 Python strftime 格式码。例如 '%Y-%m-%d' 仅返回 YYYY-MM-DD 日期，'%H:%M:%S' 仅返回 HH:MM:SS 时间，'%B %d, %Y' 返回如 'May 07, 2025'。默认为 '%Y-%m-%d %H:%M:%S'，返回完整时间戳如 '2025-05-07 14:32:15'。",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)


db_query_schema = ToolParam(
    {
        "name": "db_query",
        "description": "对 SQLite 数据库执行 SQL 查询并返回结果。该工具可在指定 SQLite 数据库上执行 SELECT、INSERT、UPDATE、DELETE 等 SQL 语句。SELECT 返回结构化查询结果；其他类型返回操作影响的元数据（如受影响行数）。工具内置防 SQL 注入措施，并以清晰错误信息处理异常。支持连接、聚合、子查询和事务等复杂查询。结果可按不同格式返回，如表格展示或结构化数据供后续处理。",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "要执行的 SQL 语句。可为任意合法 SQLite SQL，包括 SELECT、INSERT、UPDATE、DELETE、CREATE TABLE 等。",
                },
                "database_path": {
                    "type": "string",
                    "description": "SQLite 数据库文件路径。未提供时使用系统中配置的默认数据库。",
                },
                "params": {
                    "type": "object",
                    "description": "用于参数化语句的绑定参数，为字典，键对应查询中的命名参数（如查询含 ':user_id' 则 {'user_id': 123}）。强烈建议使用参数化查询以防 SQL 注入。",
                },
                "result_format": {
                    "type": "string",
                    "description": "查询结果的返回格式：'dict'（每行一个字典的列表）、'list'（首行为列名的列表的列表）、'table'（格式化为 ASCII 表格）。默认为 'dict'。",
                    "enum": ["dict", "list", "table"],
                    "default": "dict",
                },
                "max_rows": {
                    "type": "integer",
                    "description": "SELECT 查询返回的最大行数，用于限制可能返回大量数据的结果。0 表示不限制。默认为 1000。",
                    "default": 1000,
                },
                "transaction": {
                    "type": "boolean",
                    "description": "是否在事务中执行。为 true 时查询会被 BEGIN/COMMIT 包裹，出错时可回滚。SELECT 默认为 false，其他类型默认为 true。",
                    "default": False,
                },
            },
            "required": ["query"],
        },
    }
)

In [680]:
tools = [
    db_query_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
    get_current_datetime_schema,
]
messages = []

add_user_message(messages, "1+1是多少？")

chat(messages,tools=tools)

TypeError: Messages.create() got an unexpected keyword argument 'cache_control'

In [681]:
add_user_message(messages, "2+2是多少？")

chat(messages,tools=tools)

Message(id=None, content=[TextBlock(citations=None, text='4', type='text')], model='claude-sonnet-4-5', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=None, cache_read_input_tokens=None, inference_geo=None, input_tokens=2752, output_tokens=1, server_tool_use=None, service_tier=None))

In [682]:
messages = []
user_message = {
        "role": "user",
        "content": [{
          "type": "text",
          "text": code_prompt,
          "cache_control": {"type": "ephemeral"}
        }],
    }
messages.append(user_message)
chat(messages)


Message(id=None, content=[TextBlock(citations=None, text='我明白了！我是一个专门为文档分析流程生成 TypeScript 代码的代码生成器。我会在一个受限的 QuickJS 沙箱环境中工作,使用预定义的组件和全局函数来构建应用。\n\n关键要点:\n- 只能使用提供的全局函数(setState, getState, callLLM, navigateTo等)\n- 使用预定义组件构建UI,不能用标准HTML或React\n- 必须实现 getInitialState() 和 render() 函数\n- 用 schema 获取结构化的 LLM 响应\n- 多页面流程用 Route 和 Link 组件\n- 文档内容放在 systemPrompt 中\n- 不添加代码注释\n- 生成可直接运行的 TypeScript 代码\n\n我已准备好根据你的需求生成代码。请告诉我你想要构建什么样的文档分析流程?', type='text')], model='claude-sonnet-4-5', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=None, cache_read_input_tokens=None, inference_geo=None, input_tokens=8339, output_tokens=223, server_tool_use=None, service_tier=None))

In [ ]:
messages = []
user_message = {
        "role": "user",
        "content": [{
          "type": "text",
          "text": code_prompt,
          "cache_control": {"type": "ephemeral"}
        }],
    }
messages.append(user_message)
chat(messages)


Message(id=None, content=[TextBlock(citations=None, text='我明白了！我是一个专门为文档分析流程生成 TypeScript 代码的代码生成器。我会在一个受限的 QuickJS 沙箱环境中工作,使用预定义的组件和全局函数来构建应用。\n\n关键要点:\n- 只能使用提供的全局函数(setState, getState, callLLM, navigateTo等)\n- 使用预定义组件构建UI,不能用标准HTML或React\n- 必须实现 getInitialState() 和 render() 函数\n- 用 schema 获取结构化的 LLM 响应\n- 多页面流程用 Route 和 Link 组件\n- 文档内容放在 systemPrompt 中\n- 不添加代码注释\n- 生成可直接运行的 TypeScript 代码\n\n我已准备好根据你的需求生成代码。请告诉我你想要构建什么样的文档分析流程?', type='text')], model='claude-sonnet-4-5', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=None, cache_read_input_tokens=None, inference_geo=None, input_tokens=8339, output_tokens=223, server_tool_use=None, service_tier=None))